# Programming Assignment: Duplicate Detection & Removal

Welcome to the **Duplicate Detection & Removal** assignment!

Duplicate records are a common data quality problem. They arise from data entry errors, merged data sources, repeated form submissions, and system migrations. Left untreated, duplicates distort statistics, bias model training, and inflate evaluation metrics.

There are two main flavours of duplicates:

| Type | Description | Example |
|------|-------------|---------|
| **Exact duplicates** | Every field is identical | Same row copied twice |
| **Near-duplicates** | Most fields match; minor variation in one field | "Alice Smith" vs "ALICE SMITH" |

**When to remove duplicates:**
- Before model training – duplicates in the train set can cause data leakage into a test set.
- Before aggregations – counts, means, and sums are inflated by repeated rows.
- Before joining with external data – duplicate keys cause row multiplication.

**What You Will Do in This Assignment:**

* Detect exact duplicate rows using pandas built-in methods
* Remove exact duplicates while preserving the original row order
* Identify near-duplicate strings with a fuzzy matching score
* Flag duplicate rows with a boolean column for downstream use
* Produce a structured deduplication report with key metrics

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

---

## Table of Contents
- [Imports](#imports)
- [1 - Understanding the Data](#1---understanding-the-data)
- [2 - Exact Duplicate Detection](#2---exact-duplicate-detection)
    - **[Exercise 1 - `detect_exact_duplicates`](#exercise-1---detect_exact_duplicates)**
- [3 - Removing Exact Duplicates](#3---removing-exact-duplicates)
    - **[Exercise 2 - `remove_exact_duplicates`](#exercise-2---remove_exact_duplicates)**
- [4 - Fuzzy / Near-Duplicate Detection](#4---fuzzy--near-duplicate-detection)
    - **[Exercise 3 - `detect_fuzzy_duplicates`](#exercise-3---detect_fuzzy_duplicates)**
- [5 - Flagging Duplicates](#5---flagging-duplicates)
    - **[Exercise 4 - `flag_duplicates`](#exercise-4---flag_duplicates)**
- [6 - Deduplication Report](#6---deduplication-report)
    - **[Exercise 5 - `create_deduplication_report`](#exercise-5---create_deduplication_report)**
- [7 - Visualizing Results](#7---visualizing-results)

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd
import difflib

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-the-data'></a>
## 1 - Understanding the Data

We generate a synthetic customer dataset that already contains **exact duplicates** (rows copied verbatim) and **near-duplicates** (rows where only the `name` field has minor casing or spacing variation).

This controlled setup lets you verify your detection and removal functions against known expectations.

In [ ]:
df = helper_utils.generate_customer_dataset(n_samples=500, duplicate_rate=0.15, random_state=42)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn dtypes:")
print(df.dtypes)
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
helper_utils.visualize_duplicate_summary(df)

<a name='2---exact-duplicate-detection'></a>
## 2 - Exact Duplicate Detection

An **exact duplicate** is a row where **every column value** is identical to another row. `pandas` provides `DataFrame.duplicated()` to identify such rows.

Key parameter: `keep`
- `keep='first'` — marks all occurrences **except the first** as duplicates
- `keep='last'`  — marks all occurrences **except the last** as duplicates
- `keep=False`   — marks **all** occurrences (including the first) as duplicates

<a name='exercise-1---detect_exact_duplicates'></a>
### **Exercise 1 - `detect_exact_duplicates`**

**Your Task:**

Implement `detect_exact_duplicates(df)` that returns a DataFrame containing **all rows** that are part of an exact duplicate group (i.e., every occurrence, not just the second one).

**Requirements:**
- Return a `pd.DataFrame` whose rows are the duplicate rows
- Use `keep=False` so every occurrence of a repeated row is included
- Do not modify the original DataFrame

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step 1:** Use `df.duplicated(keep=False)` — this returns a boolean Series where `True` means the row has at least one duplicate somewhere in the DataFrame.

**Step 2:** Use boolean indexing: `df[mask]` to select the rows where the mask is `True`.

**Step 3:** `.copy()` the result to avoid returning a view.

</details>

In [ ]:
# GRADED FUNCTION: detect_exact_duplicates
def detect_exact_duplicates(df):
    """
    Return all rows that are part of an exact duplicate group.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.DataFrame: Subset of df where every row appears more than once
                      (all occurrences included).
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
exact_dupes = detect_exact_duplicates(df)
print(f"Total rows          : {len(df)}")
print(f"Exact duplicate rows: {len(exact_dupes)}")
print(f"\nSample of detected duplicates:")
exact_dupes.head(10)

In [ ]:
# Test your code!
unittests.exercise_1(detect_exact_duplicates)

<a name='3---removing-exact-duplicates'></a>
## 3 - Removing Exact Duplicates

After detecting duplicates, the natural next step is to remove them. `DataFrame.drop_duplicates()` does this efficiently.

Best practice: **reset the index** after dropping rows so that subsequent integer-based access (`df.iloc`) behaves predictably.

<a name='exercise-2---remove_exact_duplicates'></a>
### **Exercise 2 - `remove_exact_duplicates`**

**Your Task:**

Implement `remove_exact_duplicates(df, keep='first')` that removes all exact duplicate rows and returns the cleaned DataFrame.

**Requirements:**
- Accept `keep` as `'first'`, `'last'`, or `False`
- Return a `pd.DataFrame` with no remaining exact duplicates
- Reset the index so it runs from 0 to `n-1`
- Do not modify the original DataFrame

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step 1:** `df.drop_duplicates(keep=keep)` removes duplicate rows according to the `keep` strategy.

**Step 2:** `.reset_index(drop=True)` resets the integer index without adding the old index as a column.

</details>

In [ ]:
# GRADED FUNCTION: remove_exact_duplicates
def remove_exact_duplicates(df, keep='first'):
    """
    Remove exact duplicate rows from a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.
        keep (str): Which occurrence to keep: 'first', 'last', or False.

    Returns:
        pd.DataFrame: Deduplicated DataFrame with a reset integer index.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
df_cleaned = remove_exact_duplicates(df)
print(f"Original shape : {df.shape}")
print(f"Cleaned shape  : {df_cleaned.shape}")
print(f"Rows removed   : {len(df) - len(df_cleaned)}")
print(f"Duplicates remaining: {df_cleaned.duplicated().sum()}")

In [ ]:
# Test your code!
unittests.exercise_2(remove_exact_duplicates)

<a name='4---fuzzy--near-duplicate-detection'></a>
## 4 - Fuzzy / Near-Duplicate Detection

Exact matching misses records that are **almost the same** but differ in capitalization, spacing, or minor typos. **Fuzzy matching** computes a similarity score between two strings and flags pairs above a chosen threshold.

Python's standard library module `difflib.SequenceMatcher` provides a `ratio()` method that returns a value in $[0, 1]$ representing how similar two strings are. We scale it to $[0, 100]$ for readability.

<a name='exercise-3---detect_fuzzy_duplicates'></a>
### **Exercise 3 - `detect_fuzzy_duplicates`**

**Your Task:**

Implement `detect_fuzzy_duplicates(df, column, threshold=80)` that returns a DataFrame of all pairs of rows whose string values in `column` are similar enough (score ≥ `threshold`).

**Requirements:**
- Compare every pair of non-null values in `column` (case-insensitive)
- Use `difflib.SequenceMatcher` to compute similarity (0–100 scale)
- Return a `pd.DataFrame` with columns: `index_1`, `index_2`, `value_1`, `value_2`, `similarity`
- Only include pairs where `similarity >= threshold`

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step 1:** Extract unique values:  
```python
values = df[column].dropna().astype(str).tolist()
```

**Step 2:** Use a nested loop over indices `i` and `j` (where `j > i`) to avoid comparing a row with itself or repeating pairs.

**Step 3:** Compute the ratio:
```python
ratio = difflib.SequenceMatcher(None, values[i].lower(), values[j].lower()).ratio() * 100
```

**Step 4:** If `ratio >= threshold`, append a dict `{'index_1': i, 'index_2': j, 'value_1': values[i], 'value_2': values[j], 'similarity': ratio}` to a list.

**Step 5:** `pd.DataFrame(pairs)` converts the list to a DataFrame.

</details>

In [ ]:
# GRADED FUNCTION: detect_fuzzy_duplicates
def detect_fuzzy_duplicates(df, column, threshold=80):
    """
    Detect near-duplicate pairs in a string column using fuzzy matching.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Name of the string column to compare.
        threshold (float): Minimum similarity score (0–100) to include a pair.

    Returns:
        pd.DataFrame: DataFrame with columns index_1, index_2, value_1,
                      value_2, similarity for each pair above the threshold.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
# Run on a small sample for speed
df_sample = df.head(50).copy()
fuzzy_pairs = detect_fuzzy_duplicates(df_sample, column='name', threshold=80)
print(f"Fuzzy duplicate pairs found: {len(fuzzy_pairs)}")
if len(fuzzy_pairs) > 0:
    print(fuzzy_pairs.sort_values('similarity', ascending=False).head(10).to_string(index=False))

In [ ]:
# Test your code!
unittests.exercise_3(detect_fuzzy_duplicates)

<a name='5---flagging-duplicates'></a>
## 5 - Flagging Duplicates

Sometimes you do **not** want to remove duplicates immediately. Instead, you add a **boolean flag column** so downstream steps can decide what to do. This is safer in production pipelines where silent data loss can be dangerous.

<a name='exercise-4---flag_duplicates'></a>
### **Exercise 4 - `flag_duplicates`**

**Your Task:**

Implement `flag_duplicates(df)` that returns the DataFrame with a new boolean column `is_duplicate`. The first occurrence of each group should be `False`; all subsequent occurrences should be `True`.

**Requirements:**
- Return a `pd.DataFrame` with exactly one additional column: `is_duplicate` (bool)
- The number of rows must remain the same as the input
- Use `keep='first'` semantics — the first occurrence is **not** flagged
- Do not modify the original DataFrame

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step 1:** `result = df.copy()` to avoid modifying the original.

**Step 2:** `df.duplicated(keep='first')` returns a boolean Series — `True` for every row that is a later-occurring duplicate.

**Step 3:** Assign it: `result['is_duplicate'] = df.duplicated(keep='first')`

</details>

In [ ]:
# GRADED FUNCTION: flag_duplicates
def flag_duplicates(df):
    """
    Add a boolean column 'is_duplicate' marking later-occurring duplicates.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.DataFrame: Copy of df with an additional boolean column 'is_duplicate'.
                      The first occurrence of each group is False; repeats are True.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
df_flagged = flag_duplicates(df)
print(f"Shape              : {df_flagged.shape}")
print(f"Rows flagged True  : {df_flagged['is_duplicate'].sum()}")
print(f"Rows flagged False : {(~df_flagged['is_duplicate']).sum()}")
print(f"\nSample flagged rows:")
df_flagged[df_flagged['is_duplicate']].head(5)

In [ ]:
# Test your code!
unittests.exercise_4(flag_duplicates)

<a name='6---deduplication-report'></a>
## 6 - Deduplication Report

A **deduplication report** summarises the state of a dataset in a machine-readable dictionary. Reports are useful for logging, monitoring data quality over time, and triggering alerts when duplicate rates exceed thresholds.

<a name='exercise-5---create_deduplication_report'></a>
### **Exercise 5 - `create_deduplication_report`**

**Your Task:**

Implement `create_deduplication_report(df)` that returns a dict summarising the duplicate situation in the DataFrame.

**Requirements:**
- Return a `dict` with **exactly** these keys:
  - `'exact_count'` — number of rows that are duplicates (would be removed by `drop_duplicates(keep='first')`)
  - `'total_rows'` — total number of rows in `df`
  - `'clean_rows'` — number of rows that would remain after removing exact duplicates
- All values must be plain Python `int`

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step 1:** `df.duplicated().sum()` counts rows that are duplicates (i.e., would be removed). Wrap it with `int()` to ensure a plain Python integer.

**Step 2:** `len(df)` gives `total_rows`.

**Step 3:** `clean_rows = total_rows - exact_count`.

</details>

In [ ]:
# GRADED FUNCTION: create_deduplication_report
def create_deduplication_report(df):
    """
    Create a summary report of duplicates in the DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        dict: Dictionary with keys 'exact_count', 'total_rows', 'clean_rows'.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
report = create_deduplication_report(df)
print("Deduplication Report")
print("=" * 30)
for key, value in report.items():
    print(f"  {key:<20}: {value}")

In [ ]:
# Test your code!
unittests.exercise_5(create_deduplication_report)

<a name='7---visualizing-results'></a>
## 7 - Visualizing Results

Finally, let's compare the distributions of key numeric columns before and after deduplication to confirm that removing duplicates did not significantly distort the underlying data.

In [ ]:
helper_utils.plot_duplicate_distribution(df_cleaned, df)

---

## Congratulations! 🎉

You have successfully completed the **Duplicate Detection & Removal** assignment! Here's a summary of what you implemented:

| Exercise | Function | Key Concept |
|----------|----------|-------------|
| 1 | `detect_exact_duplicates` | `duplicated(keep=False)` |
| 2 | `remove_exact_duplicates` | `drop_duplicates()` + index reset |
| 3 | `detect_fuzzy_duplicates` | `difflib.SequenceMatcher` |
| 4 | `flag_duplicates` | Boolean flag column `is_duplicate` |
| 5 | `create_deduplication_report` | Structured dict summary |

These skills are essential for building robust data pipelines and ensuring high data quality before model training.